# Yantra NZ — Multimodal RAG over the NZ Government AI Corpus (DGX Spark / Ollama)

*Local, data-sovereign multimodal retrieval-augmented generation. No cloud API.*

Adapted from IBM's [Granite Multimodal RAG recipe](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/RAG/Granite_Multimodal_RAG.ipynb).
The original serves the vision and generation models on Replicate (a paid cloud
service). This version runs **everything locally on an NVIDIA DGX Spark (GB10)**
via **Ollama**, so document content never leaves the machine — the data-residency
story that matters for NZ public-sector work (Health NZ, Auckland Council, etc.).

**Pipeline:** Docling parses each PDF → text, tables, and images.
A local Granite **vision** model captions the images. A local Granite
**embedding** model vectorises everything into ChromaDB. A local Granite
**generation** model answers questions over the retrieved context.

**Corpus:** real NZ government AI policy/strategy/guidance documents (see
`corpus_sources.py`). Run `python scripts/fetch_corpus.py` first.

## Step 0: Configuration

Set model tags to match what `ollama list` shows on your Spark.

In [1]:
import os

# Ollama model tags — adjust to the tags actually pulled on your Spark.
GEN_MODEL = os.environ.get("GEN_MODEL", "granite4.1:30b")
VISION_MODEL = os.environ.get("VISION_MODEL", "granite3.2-vision")
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

# Local embedding model (runs via sentence-transformers / HF, on-device).
EMBEDDINGS_MODEL_PATH = "ibm-granite/granite-embedding-311m-multilingual-r2"

CORPUS_DIR = os.path.join(os.getcwd(), "corpus")
print("Generation model:", GEN_MODEL)
print("Vision model:    ", VISION_MODEL)
print("Embeddings:      ", EMBEDDINGS_MODEL_PATH)
print("Corpus dir:      ", CORPUS_DIR)

Generation model: granite4.1:30b
Vision model:     granite3.2-vision
Embeddings:       ibm-granite/granite-embedding-311m-multilingual-r2
Corpus dir:       /home/sailendhran/Downloads/yantra-rag-eval/yantra-nz-rag/corpus


## Step 1: Load the models (all local)

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model = HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL_PATH)
embeddings_tokenizer = AutoTokenizer.from_pretrained(EMBEDDINGS_MODEL_PATH)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

In [3]:
from langchain_ollama import ChatOllama

# Vision model for image understanding (replaces Replicate granite-vision).
vision_model = ChatOllama(
    model=VISION_MODEL,
    base_url=OLLAMA_BASE_URL,
    num_predict=512,
    temperature=0,
)

# Generation model for the RAG answer step (replaces Replicate granite-4.1-8b).
model = ChatOllama(
    model=GEN_MODEL,
    base_url=OLLAMA_BASE_URL,
    num_predict=1000,
    temperature=0,
)

## Step 2: Discover the corpus

Reads every PDF in `./corpus/`. If empty, run `python scripts/fetch_corpus.py`
(or drop PDFs in manually if a download was blocked by a CDN).

In [4]:
import glob

sources = sorted(glob.glob(os.path.join(CORPUS_DIR, "*.pdf")))
assert sources, (
    f"No PDFs in {CORPUS_DIR}. Run `python scripts/fetch_corpus.py` first, "
    "or place NZ govt AI PDFs there manually."
)
print(f"{len(sources)} document(s) in corpus:")
for s in sources:
    print("  -", os.path.basename(s))

20 document(s) in corpus:
  - 1cdad902-en.pdf
  - 2d1956f0-en.pdf
  - 5fd44e6c-en.pdf
  - 718c93a1-en.pdf
  - A-guide-for-policy-makers_AI.pdf
  - AI-and-the-Information-Privacy-Principles.pdf
  - Algorithm-Charter-2020_Final-English-1.pdf
  - Lawyers-and-AI-Guidance-Mar-2024.pdf
  - Procurement Applications - AI Governance.pdf
  - Procurement Data - AI Governance.pdf
  - Procurement Models - AI Governance.pdf
  - Public-Service-AI-work-programme-to-2027-A3.v1.pdf
  - Public-Service-Artifical-Intelligence-Framework.pdf
  - Responsible-AI-Guidance-for-the-Public-Service-GenAI-Print.pdf
  - Responsible-AI-in-action-Guidance.pdf
  - Trustworthy-AI-in-Aotearoa-March-2020.pdf
  - ai-procurement-checklist.pdf
  - approach-to-work-on-artificial-intelligence.pdf
  - new-zealands-strategy-for-artificial-intelligence.pdf
  - responsible-ai-guidance-for-businesses.pdf


## Step 3: Parse documents with Docling (text, tables, images)

In [5]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

pdf_pipeline_options = PdfPipelineOptions(
    do_ocr=False,                  # set True if any corpus PDF is scanned
    generate_picture_images=True,  # needed so the vision model can caption images
)
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_pipeline_options)}
)

conversions = {
    os.path.basename(src): converter.convert(source=src).document for src in sources
}
print("Parsed:", list(conversions.keys()))

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Parsed: ['1cdad902-en.pdf', '2d1956f0-en.pdf', '5fd44e6c-en.pdf', '718c93a1-en.pdf', 'A-guide-for-policy-makers_AI.pdf', 'AI-and-the-Information-Privacy-Principles.pdf', 'Algorithm-Charter-2020_Final-English-1.pdf', 'Lawyers-and-AI-Guidance-Mar-2024.pdf', 'Procurement Applications - AI Governance.pdf', 'Procurement Data - AI Governance.pdf', 'Procurement Models - AI Governance.pdf', 'Public-Service-AI-work-programme-to-2027-A3.v1.pdf', 'Public-Service-Artifical-Intelligence-Framework.pdf', 'Responsible-AI-Guidance-for-the-Public-Service-GenAI-Print.pdf', 'Responsible-AI-in-action-Guidance.pdf', 'Trustworthy-AI-in-Aotearoa-March-2020.pdf', 'ai-procurement-checklist.pdf', 'approach-to-work-on-artificial-intelligence.pdf', 'new-zealands-strategy-for-artificial-intelligence.pdf', 'responsible-ai-guidance-for-businesses.pdf']


### Text chunks

In [6]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.types.doc.document import TableItem
from langchain_core.documents import Document

doc_id = 0
texts: list[Document] = []
for source, docling_document in conversions.items():
    for chunk in HybridChunker(tokenizer=embeddings_tokenizer).chunk(docling_document):
        items = chunk.meta.doc_items
        if len(items) == 1 and isinstance(items[0], TableItem):
            continue  # tables handled separately
        refs = " ".join(item.get_ref().cref for item in items)
        texts.append(Document(
            page_content=chunk.text,
            metadata={"doc_id": (doc_id := doc_id + 1), "source": source, "ref": refs},
        ))
print(f"{len(texts)} text chunks")

1279 text chunks


### Tables (exported to markdown)

In [7]:
from docling_core.types.doc.labels import DocItemLabel

tables: list[Document] = []
for source, docling_document in conversions.items():
    for table in docling_document.tables:
        if table.label in [DocItemLabel.TABLE]:
            tables.append(Document(
                page_content=table.export_to_markdown(docling_document),
                metadata={"doc_id": (doc_id := doc_id + 1), "source": source,
                          "ref": table.get_ref().cref},
            ))
print(f"{len(tables)} tables")

123 tables


### Images (captioned by the local vision model)

Government strategy PDFs are figure-heavy (frameworks, OECD principle diagrams,
roadmaps). Capturing those as text is exactly what makes this *multimodal*.

In [8]:
import base64
import io
import PIL.Image
import PIL.ImageOps
from langchain_core.messages import HumanMessage


def encode_image(image: PIL.Image.Image, fmt: str = "png") -> str:
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, fmt)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


IMAGE_PROMPT = (
    "Describe this image from a New Zealand government AI policy document. "
    "If it contains text, diagrams, or framework structure, explain it in detail."
)

pictures: list[Document] = []
for source, docling_document in conversions.items():
    for picture in docling_document.pictures:
        image = picture.get_image(docling_document)
        if not image:
            continue
        # Ollama multimodal: pass base64 image(s) alongside the text prompt.
        msg = HumanMessage(
            content=IMAGE_PROMPT,
            additional_kwargs={"images": [encode_image(image)]},
        )
        try:
            response = vision_model.invoke([msg])
            caption = response.content
        except Exception as e:  # keep going if one image fails
            caption = f"[vision caption failed: {e}]"
        pictures.append(Document(
            page_content=caption,
            metadata={"doc_id": (doc_id := doc_id + 1), "source": source,
                      "ref": picture.get_ref().cref},
        ))
print(f"{len(pictures)} image descriptions")

418 image descriptions


## Step 4: Build the vector database (ChromaDB, local)

In [9]:
import itertools
from langchain_chroma import Chroma

vector_db = Chroma(embedding_function=embeddings_model)
documents = list(itertools.chain(texts, tables, pictures))
ids = [i for d in documents for i in vector_db.add_documents([d])]
print(f"{len(ids)} documents indexed")

1820 documents indexed


## Step 5: RAG query

Retrieve relevant chunks, then have the local Granite model answer over them.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are an assistant answering questions about New Zealand government AI "
    "policy. Use ONLY the context below. If the answer is not in the context, "
    "say so.\n\nContext:\n{context}\n\nQuestion: {input}\n\nAnswer:"
)

retriever = vector_db.as_retriever(search_kwargs={"k": 5})


def ask(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[{d.metadata['source']}] {d.page_content}" for d in docs
    )
    chain = RAG_PROMPT | model
    answer = chain.invoke({"context": context, "input": question}).content
    print("Q:", question, "\n")
    print("A:", answer, "\n")
    print("Sources:", sorted({d.metadata["source"] for d in docs}))
    return answer

### Try it — questions grounded in the NZ AI corpus

In [11]:
ask("What principles underpin New Zealand's approach to responsible AI in the public service?")

Q: What principles underpin New Zealand's approach to responsible AI in the public service? 

A: According to the provided context, New Zealand's approach to responsible AI in the public service is underpinned by the OECD AI Principles. Specifically, the "Responsible-AI-Guidance-for-the-Public-Service-GenAI-Print.pdf" states that "In June 2024, Cabinet agreed that New Zealand's approach to adopting AI will be in accordance with the OECD AI Principles. These principles promote innovative, trustworthy AI that respects human rights and democratic values." The guidance also indicates the relevant OECD AI Principles in each section, aligning the public service's use of AI with these internationally recognized standards for responsible AI development and deployment. 

Sources: ['Responsible-AI-Guidance-for-the-Public-Service-GenAI-Print.pdf', 'approach-to-work-on-artificial-intelligence.pdf', 'new-zealands-strategy-for-artificial-intelligence.pdf']


'According to the provided context, New Zealand\'s approach to responsible AI in the public service is underpinned by the OECD AI Principles. Specifically, the "Responsible-AI-Guidance-for-the-Public-Service-GenAI-Print.pdf" states that "In June 2024, Cabinet agreed that New Zealand\'s approach to adopting AI will be in accordance with the OECD AI Principles. These principles promote innovative, trustworthy AI that respects human rights and democratic values." The guidance also indicates the relevant OECD AI Principles in each section, aligning the public service\'s use of AI with these internationally recognized standards for responsible AI development and deployment.'

In [12]:
ask("How does the guidance treat the use of GenAI with sensitive or classified data?")

Q: How does the guidance treat the use of GenAI with sensitive or classified data? 

A: The provided context does not explicitly address the use of GenAI with sensitive or classified data. While the guidance discusses privacy risks, cybersecurity concerns, and the importance of robust risk assessments when using personal information in public GenAI systems, it does not specifically mention how to handle sensitive or classified data within this framework. Therefore, based on the given context, there is no direct answer to how the guidance treats the use of GenAI with sensitive or classified data. 

Sources: ['Lawyers-and-AI-Guidance-Mar-2024.pdf', 'Responsible-AI-Guidance-for-the-Public-Service-GenAI-Print.pdf']


'The provided context does not explicitly address the use of GenAI with sensitive or classified data. While the guidance discusses privacy risks, cybersecurity concerns, and the importance of robust risk assessments when using personal information in public GenAI systems, it does not specifically mention how to handle sensitive or classified data within this framework. Therefore, based on the given context, there is no direct answer to how the guidance treats the use of GenAI with sensitive or classified data.'

In [13]:
ask("How are Treaty of Waitangi and Māori perspectives reflected in NZ's AI strategy?")

Q: How are Treaty of Waitangi and Māori perspectives reflected in NZ's AI strategy? 

A: Based on the provided context, Treaty of Waitangi obligations and Māori perspectives are reflected in New Zealand's AI strategy in the following ways:

1. **Embedding Te Ao Māori Perspective**: The Algorithm Charter (2020) explicitly states the commitment to "embed a Te Ao Māori perspective in the development and use of algorithms consistent with the principles of the Treaty of Waitangi." This indicates that Māori worldviews and values are to be integrated into AI systems to align with Treaty commitments.

2. **Engagement with Data Iwi Leaders' Group (DILG)**: The mention of meetings with the DILG and their "ready uptake of AI" suggests active engagement with Māori leaders in shaping AI policy. The concern expressed about the danger of Māori being left outside AI's potential benefits highlights a proactive approach to ensuring Māori participation and advantage from AI technologies.

3. **Public Ben

'Based on the provided context, Treaty of Waitangi obligations and Māori perspectives are reflected in New Zealand\'s AI strategy in the following ways:\n\n1. **Embedding Te Ao Māori Perspective**: The Algorithm Charter (2020) explicitly states the commitment to "embed a Te Ao Māori perspective in the development and use of algorithms consistent with the principles of the Treaty of Waitangi." This indicates that Māori worldviews and values are to be integrated into AI systems to align with Treaty commitments.\n\n2. **Engagement with Data Iwi Leaders\' Group (DILG)**: The mention of meetings with the DILG and their "ready uptake of AI" suggests active engagement with Māori leaders in shaping AI policy. The concern expressed about the danger of Māori being left outside AI\'s potential benefits highlights a proactive approach to ensuring Māori participation and advantage from AI technologies.\n\n3. **Public Benefit through Treaty Commitments**: The Algorithm Charter emphasizes delivering 

---
**Yantra NZ note.** This is a reusable capability asset: a local, data-sovereign
multimodal RAG over the live NZ public-sector AI landscape, running on GB10
hardware. It doubles as (a) a yantranz.com case study — *on-prem RAG for
regulated NZ data* — and (b) a watsonx-adjacent demo you can re-point at any
agency's document set. To extend: add PDFs to `corpus/`, register them in
`corpus_sources.py`, re-run from Step 2.

## Step 6: Generate a sourced PDF report (with tables, charts, and images)

This turns the RAG pipeline into a **document generator**. It writes a polished,
branded PDF where:
- each narrative section is generated by the local model over *retrieved* context
  and lists the sources it drew on;
- a data table and bar chart are built from quantitative figures **extracted** from
  the corpus (never invented) — each row carries its source;
- figures from the source PDFs (Docling-extracted) are re-placed with captions;
- a References section lists every source used.

Everything runs locally on the Spark. This is the Yantra capability demo: an
on-prem, auditable briefing assembled from public NZ government documents.

In [14]:
from report_generator import ReportContext, generate_report, DEFAULT_SECTIONS

ctx = ReportContext(
    retriever=retriever,        # built in Step 5
    model=model,                # ChatOllama generation model from Step 1
    conversions=conversions,    # Docling docs from Step 3
    pictures=pictures,          # image captions from Step 3
    workdir=os.getcwd(),
    extract_images=True,
)

pdf_path = generate_report(
    ctx,
    title="The New Zealand Government AI Landscape",
    subtitle="A sourced briefing generated on-premises from public sector documents",
    output_filename="NZ_Government_AI_Landscape.pdf",
)
print("Report written to:", pdf_path)

  · writing section: Executive Summary
  · writing section: Guiding Principles
  · writing section: Public Service Governance & Assurance
  · writing section: Treaty of Waitangi & Māori Perspectives
  · writing section: Economic Opportunity & Adoption
  · extracting quantitative figures for table/chart
  ✓ wrote /home/sailendhran/Downloads/yantra-rag-eval/yantra-nz-rag/NZ_Government_AI_Landscape.pdf
Report written to: /home/sailendhran/Downloads/yantra-rag-eval/yantra-nz-rag/NZ_Government_AI_Landscape.pdf


### Tips for a sharper report
- **Keep charts to one metric type.** Set `figure_query` to target a single kind
  of number (e.g. "AI adoption percentages by year in New Zealand") so the chart
  axis is consistent — mixing % and $ on one axis is misleading.
- **Customise sections.** Pass your own `sections=[Section("Heading", "query"), ...]`
  to `generate_report` to retarget the briefing (e.g. for a specific agency).
- **Re-point the corpus.** Swap the PDFs in `corpus/` and this same pipeline
  produces a briefing on any document set — that's the reusable Yantra asset.

# Step 7: Observability with Langfuse (self-hosted, local)

Capture a full trace of each RAG call — retrieval (query, sources, latency) and
generation (prompt, model, output, token counts) — viewable in a local dashboard.

**One-time setup (in a terminal on the Spark):**
```bash
docker compose -f docker-compose.langfuse.yml up -d   # starts Langfuse at :3000
# open http://localhost:3000, create an account + project, copy the API keys
cp .env.example .env                                   # paste keys into .env
```
Then restart the kernel so the keys load, or set them inline below.

In [17]:
# Load Langfuse keys from .env if present (so traced_rag emits to the dashboard).
from dotenv import load_dotenv, find_dotenv
print("CWD:", os.getcwd())
print("find_dotenv() ->", repr(find_dotenv()))
print(".env exists in CWD:", os.path.exists(".env"))

load_dotenv(find_dotenv(), override=True)
import os
print("LANGFUSE_BASE_URL ->", os.environ.get("LANGFUSE_BASE_URL"))
print("Langfuse host:", os.environ.get("LANGFUSE_HOST", "(not set — tracing will no-op)"))

CWD: /home/sailendhran/Downloads/yantra-rag-eval/yantra-nz-rag
find_dotenv() -> '/home/sailendhran/Downloads/yantra-rag-eval/yantra-nz-rag/.env'
.env exists in CWD: True
LANGFUSE_BASE_URL -> http://localhost:3000
Langfuse host: (not set — tracing will no-op)


In [18]:
from traced_rag import TracedRAG

trag = TracedRAG(retriever, model, gen_model_name=GEN_MODEL, k=5)

# Same questions as before — but now each call produces a trace you can inspect.
trag.ask("What principles underpin New Zealand's approach to responsible AI in the public service?")
trag.ask("How does the guidance treat the use of GenAI with sensitive or classified data?")
trag.ask("How are Treaty of Waitangi and Māori perspectives reflected in NZ's AI strategy?")

Q: What principles underpin New Zealand's approach to responsible AI in the public service? 

A: Based on the provided context, New Zealand's approach to responsible AI in the public service is underpinned by the following principles:

1. **Alignment with OECD AI Principles**: Cabinet agreed in June 2024 that New Zealand's approach to adopting AI will be in accordance with the OECD AI Principles. These principles promote innovative, trustworthy AI that respects human rights and democratic values.

2. **Responsible Development and Use**: The context emphasizes "responsible development and use of AI in New Zealand" to deliver better outcomes for people, boost productivity, contribute to economic prosperity, improve public services, meet government targets, and ensure the safety of the country and its people.

3. **Encouragement of AI Adoption with Risk Management**: There is an agreement that "government agencies should be encouraged to adopt AI for its benefits, while managing the risks

RAGResult(question="How are Treaty of Waitangi and Māori perspectives reflected in NZ's AI strategy?", answer='Based on the provided context, New Zealand\'s AI strategy reflects Treaty of Waitangi obligations and Māori perspectives in the following ways:\n\n1. **Embedding Te Ao Māori Perspective**: The strategy explicitly aims to "embed a Te Ao Māori perspective in the development and use of algorithms consistent with the principles of the Treaty of Waitangi." This indicates a commitment to incorporating Māori worldviews and values into AI systems to ensure they align with Treaty commitments.\n\n2. **Engagement with Data Iwi Leaders\' Group (DILG)**: The mention of recent meetings with the DILG and their "ready uptake of AI" suggests active engagement with Māori leaders in shaping AI policy. This collaboration acknowledges the importance of Māori participation and leadership in AI development to prevent Māori from being excluded from potential benefits.\n\n3. **Delivering Public Benefi

Open **http://localhost:3000** → your project → *Traces*. Each `rag-query` trace
has nested `retrieval` and `generation` spans with latency and (where Ollama
reports it) token counts. This is the observability view.

## Step 8: Evaluation with DeepEval (local Granite judge)

Score answer quality with a local judge model — no data leaves the Spark.

- **Reference-free:** Faithfulness (no hallucination beyond context) +
  Answer Relevancy. Works from `goldens.yaml` questions alone.
- **Golden set:** add `expected_output` to goldens to unlock Contextual
  Recall/Precision (retrieval quality).

**Setup:** `cp goldens.example.yaml goldens.yaml` and edit. Fill in
`expected_output` over time (aim for ~50 curated questions for stable metrics).

In [24]:
from evaluate import RAGEvaluator

# Judge with the same local Granite model (temperature 0 for stable scoring).
ev = RAGEvaluator(trag, judge_model=GEN_MODEL, base_url=OLLAMA_BASE_URL,
                  goldens_path="goldens.yaml", threshold=0.7)

# Reference-free metrics (no answer key needed).
ev.run_reference_free()

Reference-free eval over 8 questions (judge: granite4.1:30b)
  · What principles underpin New Zealand's approach to responsible AI in t…


Output()

Output()

  · How does NZ guidance treat the use of GenAI with sensitive or classifi…


Output()

...

  · What does the Responsible AI Guidance for Businesses recommend about r…


Output()

Output()


  Summary (mean score):
    FaithfulnessMetric           0.575  (n=8)
    AnswerRelevancyMetric        0.977  (n=8)



[{'input': "What principles underpin New Zealand's approach to responsible AI in the public service?",
  'actual_output': 'Based on the provided context, New Zealand\'s approach to responsible AI in the public service is underpinned by the following principles:\n\n1. **Alignment with OECD AI Principles**: Cabinet agreed in June 2024 that New Zealand\'s approach to adopting AI will be in accordance with the OECD AI Principles. These principles promote innovative, trustworthy AI that respects human rights and democratic values.\n\n2. **Responsible Development and Use**: The context emphasizes "responsible development and use of AI in New Zealand" to deliver better outcomes for people, boost productivity, contribute to economic prosperity, improve public services, meet government targets, and ensure the safety of the country and its people.\n\n3. **Encouragement of AI Adoption with Risk Management**: There is an agreement that "government agencies should be encouraged to adopt AI for its 

In [26]:
# Retrieval metrics — only scores goldens that have an expected_output filled in.
ev.run_with_goldens()

No goldens with expected_output found — fill them in goldens.yaml first.


[]

In [27]:
ev.save_results("eval_results.json")

Saved 8 eval rows to eval_results.json


'eval_results.json'

### Reading the scores
- **Faithfulness** low → the model is adding claims not supported by retrieved
  context (hallucination). Tighten the prompt or improve retrieval.
- **Answer Relevancy** low → answers drift off-question. Check chunking / prompt.
- **Contextual Recall** low → retrieval is missing needed passages. Raise `k`,
  revisit chunk size, or check the embedding model.
- **Contextual Precision** low → relevant chunks are buried under noise. Consider
  reranking or a smaller, cleaner `k`.

A caveat worth stating in any client demo: the judge is itself an LLM, so treat
these as directional signal. The golden set is what makes scores comparable across
pipeline changes — version it alongside your model and corpus.

---
**Yantra NZ note.** Steps 7-8 turn the demo from "it answers questions" into
"it answers questions, here are the traces, and here are faithfulness/relevancy
scores." That observability + evaluation story is what separates a production-grade
RAG from a prototype — and a strong talking point for public-sector bids where
auditability and data residency are requirements.